# Dataset baseline: nvpdyf-bdd100k

This notebook covers the first slice of the project (see project README):

1. locate + validate the dataset (it already ships in YOLO format - no
   conversion needed, unlike the older `src/datasets/bdd100k.py` which
   converts raw BDD100K json into YOLO format);
2. basic dataset info - per-split counts, class distribution, data-quality
   checks (missing/empty/orphan labels);
3. draw a couple of scenes with their ground-truth boxes, as a sanity check
   before anything gets trained on this data.

All the actual logic lives in [`src/datasets/nvpdyf_bdd100k.py`](src/datasets/nvpdyf_bdd100k.py)
and [`src/utils/visualize.py`](src/utils/visualize.py) - this notebook
just calls it and shows the results. Keeping the logic in the plain `.py`
module (instead of inline in the notebook) is why it comes with docstrings
and can be reused later from `train.py`, unit-tested, etc.


In [ ]:
import logging
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from src.datasets import nvpdyf_bdd100k as ds
from src.utils.visualize import draw_ground_truth

# the project normally logs through Hydra (see train.py); here we just want
# logger.info(...) calls to print inline, so wire the root logger to stdout
logging.basicConfig(level=logging.INFO, format="%(message)s")


## 1. Locate the dataset

Same convention as the rest of the project (see README "Kaggle notebook"
section): point `input_dir` at the whole mount root, not at a specific
dataset subfolder - Kaggle's exact path varies, and the search below is
recursive so it finds the dataset either way.

- On Kaggle: `/kaggle/input` (default below).
- Locally: override `INPUT_DIR` with wherever you downloaded
  `nvpdyf-bdd100k` to, or point it at `src/configs/datasets/nvpdyf_bdd100k.yaml`'s
  `input_dir` default.


In [ ]:
# default input_dir, read from the same Hydra config train.py would use
_cfg = OmegaConf.load("src/configs/datasets/nvpdyf_bdd100k.yaml")

INPUT_DIR = Path("/kaggle/input")  # <- override this for a local run
if not INPUT_DIR.exists():
    INPUT_DIR = Path(_cfg.input_dir)

DATA_ROOT = ds.find_dataset_root(INPUT_DIR)
CLASSES = ds.load_classes(DATA_ROOT)

print(f"dataset root: {DATA_ROOT}")
print(f"classes ({len(CLASSES)}): {CLASSES}")


## 2. Dataset info

Per-split frame/box counts and a couple of data-quality checks that are
worth knowing about *before* training:

- **missing_labels** - images with no matching `.txt` (ultralytics treats
  these as unlabeled, not empty - silently different from an intentional
  "no objects" frame);
- **empty_labels** - `.txt` files with zero lines (a frame with no objects
  in it, e.g. an empty road - legitimate, just worth knowing the count);
- **orphan_labels** - `.txt` files with no matching image (dead weight,
  usually a sign of a packaging issue upstream).


In [ ]:
SPLITS = [s for s in ("train", "val", "test") if (DATA_ROOT / "images" / s).is_dir()]

stats = {split: ds.collect_stats(DATA_ROOT, split) for split in SPLITS}
ds.describe_dataset(stats, CLASSES)


### Class distribution

One bar per class, summed over all splits, sorted descending. A single
color is enough here - the bars themselves rank the classes, so a
multi-hue categorical palette would only add noise (color should encode
identity or magnitude, never both at once).


In [ ]:
total_counts = Counter()
for s in stats.values():
    total_counts.update(s["class_counts"])

# include every class from data.yaml, even ones with 0 boxes - a class
# missing from the dataset entirely is easy to miss if it just drops out
# of the plot, a zero-length bar for it is not
items = sorted(
    ((cid, total_counts[cid]) for cid in CLASSES), key=lambda kv: kv[1]
)  # ascending, for barh
labels = [CLASSES[cid] for cid, _ in items]
values = [v for _, v in items]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(labels, values, color="#4C72B0")
ax.set_xlabel("boxes (all splits)")
ax.set_title("Class distribution - nvpdyf-bdd100k")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="0.9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for bar, v in zip(bars, values):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f" {v}", va="center", fontsize=9)
plt.tight_layout()
plt.show()


## 3. A couple of scenes

Two random frames (one from `train`, one from `val`) with their
ground-truth boxes drawn - a quick sanity check that images and labels
actually line up before anything gets trained on this data. Reruns with
the same `seed` are deterministic.


In [ ]:
SEED = _cfg.sample_seed

scenes = []
for split in ("train", "val"):
    if split in SPLITS:
        scenes += ds.sample_scenes(DATA_ROOT, split, n=1, seed=SEED)

fig = draw_ground_truth(scenes, CLASSES)
